In [4]:
import os, re, time, random
import pandas as pd
import requests
from bs4 import BeautifulSoup
from difflib import SequenceMatcher


In [7]:
import os, re, time, random
import pandas as pd
import requests
from bs4 import BeautifulSoup
from difflib import SequenceMatcher

# ── CONFIG ──────────────────────────────────────────────────────────────
INPUT_BOOKS = "pen_unique_books.csv"

IDS_OUT = "goodreads_ids_all.csv"
ENG_OUT = "goodreads_engagement_all.csv"

CHECKPOINT_EVERY = 50
SLEEP_MIN = 2.5
SLEEP_MAX = 5.0

MAX_RETRIES = 3          # retry on transient failures / rate limits
BACKOFF_BASE = 10        # seconds; doubles each retry

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"
)

BAD_TITLE = re.compile(
    r"(study guide|cliffsnotes|sparknotes|supersummary|workbook"
    r"|analysis of|summary of|teacher.?s guide|lesson plan)",
    re.I,
)


# ── HELPERS ─────────────────────────────────────────────────────────────

def norm(s: str) -> str:
    """Normalize a string for fuzzy comparison."""
    s = str(s or "").lower()
    s = re.sub(r"\(.*?\)", "", s)          # remove parentheticals
    s = re.sub(r"[^a-z0-9 ]+", " ", s)    # only alphanumerics
    s = re.sub(r"\s+", " ", s).strip()
    return s


def sim(a: str, b: str) -> float:
    return SequenceMatcher(None, norm(a), norm(b)).ratio()


def jitter_sleep():
    time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))


def backoff_sleep(attempt: int):
    """Exponential backoff: 10s, 20s, 40s, ..."""
    wait = BACKOFF_BASE * (2 ** attempt) + random.uniform(0, 3)
    print(f"  ⏳ backing off {wait:.0f}s (attempt {attempt + 1}/{MAX_RETRIES})")
    time.sleep(wait)


# ── SESSION SETUP ───────────────────────────────────────────────────────

session = requests.Session()
session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.goodreads.com/",
})


def safe_get(url: str, timeout: int = 25):
    """GET with retry + exponential backoff on transient errors."""
    for attempt in range(MAX_RETRIES):
        try:
            r = session.get(url, timeout=timeout)
            if r.status_code == 200:
                return r
            if r.status_code in (429, 503):
                # rate-limited or temporarily unavailable → retry
                print(f"  ⚠️  HTTP {r.status_code} for {url}")
                backoff_sleep(attempt)
                continue
            # other non-200 (404, 403, etc.) → don't retry
            return r
        except (requests.ConnectionError, requests.Timeout) as e:
            print(f"  ⚠️  Network error: {e}")
            if attempt < MAX_RETRIES - 1:
                backoff_sleep(attempt)
            else:
                return None
    return None


# ── PHASE 1: FIND GOODREADS IDs ────────────────────────────────────────

def extract_book_id(href: str):
    """Safely extract numeric book ID from a Goodreads href like /book/show/12345-title."""
    m = re.search(r"/book/show/(\d+)", href)
    return m.group(1) if m else None


def find_best_goodreads_id(title: str, author: str | None, max_candidates: int = 10):
    q = f"{title} {author or ''}".strip()
    url = "https://www.goodreads.com/search?q=" + requests.utils.quote(q)

    r = safe_get(url)
    if r is None:
        return None, None, {"reason": "network_error"}
    if r.status_code != 200:
        return None, None, {"http_status": r.status_code, "reason": "search_non_200"}

    soup = BeautifulSoup(r.text, "html.parser")
    candidates = []

    for a_tag in soup.select("a.bookTitle")[:max_candidates]:
        cand_title = a_tag.get_text(strip=True)
        if BAD_TITLE.search(cand_title):
            continue

        href = a_tag.get("href", "")
        book_id = extract_book_id(href)
        if book_id is None:
            continue

        row = a_tag.find_parent("tr")
        author_a = row.select_one("a.authorName") if row else None
        cand_author = author_a.get_text(strip=True) if author_a else ""

        full_url = "https://www.goodreads.com" + href

        title_score = sim(title, cand_title)
        author_score = sim(author, cand_author) if author else 0.0
        score = 0.85 * title_score + 0.15 * author_score

        candidates.append((score, book_id, full_url, cand_title, cand_author))

    if not candidates:
        return None, None, {"reason": "no_candidates"}

    candidates.sort(reverse=True, key=lambda x: x[0])
    best = candidates[0]
    return best[1], best[2], {
        "match_score": round(best[0], 4),
        "matched_title": best[3],
        "matched_author": best[4],
        "reason": "ok",
    }


# ── PHASE 2: SCRAPE ENGAGEMENT ─────────────────────────────────────────

def parse_int_from_label(text: str, label: str):
    m = re.search(rf"([\d,]+)\s+{label}", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_count_pct(s: str):
    m = re.match(r"\s*([\d,]+)\s*\(([\d.]+)%\)\s*", str(s))
    if not m:
        return None, None
    return int(m.group(1).replace(",", "")), float(m.group(2))


def scrape_engagement_by_id(book_id: str):
    url = f"https://www.goodreads.com/book/show/{book_id}"
    r = safe_get(url)
    if r is None:
        return {"goodreads_id": book_id, "goodreads_url": url, "scrape_status": "network_error"}
    if r.status_code != 200:
        return {"goodreads_id": book_id, "goodreads_url": url,
                "http_status": r.status_code, "scrape_status": "non_200"}

    soup = BeautifulSoup(r.text, "html.parser")

    avg_tag = soup.find("div", {"class": "RatingStatistics__rating"})
    avg_rating = None
    if avg_tag:
        try:
            avg_rating = float(avg_tag.get_text(strip=True))
        except ValueError:
            pass

    meta_tag = soup.find("div", {"class": "RatingStatistics__meta"})
    meta_text = meta_tag.get_text(" ", strip=True) if meta_tag else ""
    rating_count = parse_int_from_label(meta_text, "ratings")
    review_count = parse_int_from_label(meta_text, "reviews")

    dist_tags = soup.find_all("div", {"class": "RatingsHistogram__labelTotal"})
    dist = [t.get_text(strip=True) for t in dist_tags[:5]]

    row = {
        "goodreads_id": book_id,
        "goodreads_url": url,
        "avg_rating": avg_rating,
        "rating_count": rating_count,
        "review_count": review_count,
        "scrape_status": "ok",
    }

    for i, star in enumerate([5, 4, 3, 2, 1]):
        raw = dist[i] if i < len(dist) else None
        row[f"dist_{star}_star_raw"] = raw
        c, p = parse_count_pct(raw) if raw else (None, None)
        row[f"count_{star}_star"] = c
        row[f"pct_{star}_star"] = p

    # derived metrics
    pct5 = row.get("pct_5_star")
    pct1 = row.get("pct_1_star")
    if pct5 is not None and pct1 is not None:
        row["polarization"] = round((pct5 + pct1) / 100.0, 4)
        row["rating_skew"] = round((pct5 - pct1) / 100.0, 4)

    return row


# ── PERSISTENCE ─────────────────────────────────────────────────────────

def load_or_init(path: str):
    if os.path.exists(path):
        return pd.read_csv(path)
    return None


def save_checkpoint(rows: list[dict], path: str, label: str, count: int):
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"[{label}] checkpoint saved — {count} rows total")


# ── MAIN ────────────────────────────────────────────────────────────────

def main():
    books = pd.read_csv(INPUT_BOOKS).fillna("")
    if "book_title" not in books.columns:
        raise ValueError(f"{INPUT_BOOKS} must have a 'book_title' column")
    if "author" not in books.columns:
        books["author"] = ""

    print(f"Loaded {len(books)} books from {INPUT_BOOKS}")

    # ── PHASE 1: Find IDs (resume-safe) ──
    existing_ids = load_or_init(IDS_OUT)
    if existing_ids is not None:
        done = set(
            existing_ids["book_title"].astype(str) + "||" + existing_ids["author"].astype(str)
        )
        id_rows = existing_ids.to_dict("records")
        print(f"[IDs] Resuming — {len(done)} already done")
    else:
        done = set()
        id_rows = []

    new_since_checkpoint = 0
    for idx, r in books.iterrows():
        key = str(r["book_title"]) + "||" + str(r.get("author", ""))
        if key in done:
            continue

        title = str(r["book_title"]).strip()
        author = str(r.get("author", "")).strip() or None

        gid, gurl, meta = find_best_goodreads_id(title, author, max_candidates=10)

        id_rows.append({
            "book_title": title,
            "author": author or "",
            "goodreads_id": gid,
            "goodreads_url": gurl,
            "found": bool(gid),
            **meta,
        })

        status = "✅" if gid else "❌"
        print(f"  {status} [{idx+1}/{len(books)}] {title}")

        new_since_checkpoint += 1
        if new_since_checkpoint >= CHECKPOINT_EVERY:
            save_checkpoint(id_rows, IDS_OUT, "IDs", len(id_rows))
            new_since_checkpoint = 0

        jitter_sleep()

    save_checkpoint(id_rows, IDS_OUT, "IDs", len(id_rows))
    print(f"[IDs] Phase 1 complete — {IDS_OUT}")

    # ── Reload & filter to found books ──
    ids_df = pd.read_csv(IDS_OUT)
    # Robust boolean check (handles string "True" from CSV reload)
    ids_df["found"] = ids_df["found"].astype(str).str.lower() == "true"
    ids_df = ids_df[ids_df["found"]].copy()
    ids_df["goodreads_id"] = ids_df["goodreads_id"].astype(str).str.split(".").str[0]

    found_pct = len(ids_df) / len(books) * 100 if len(books) else 0
    print(f"[IDs] {len(ids_df)}/{len(books)} books matched ({found_pct:.1f}%)")

    # ── PHASE 2: Scrape engagement (resume-safe) ──
    existing_eng = load_or_init(ENG_OUT)
    if existing_eng is not None:
        eng_done = set(existing_eng["goodreads_id"].astype(str))
        eng_rows = existing_eng.to_dict("records")
        print(f"[ENG] Resuming — {len(eng_done)} already done")
    else:
        eng_done = set()
        eng_rows = []

    new_since_checkpoint = 0
    for i, r in ids_df.iterrows():
        gid = str(r["goodreads_id"]).strip()
        if gid in eng_done:
            continue

        data = scrape_engagement_by_id(gid)
        data["book_title"] = r["book_title"]
        data["author"] = r.get("author", "")
        eng_rows.append(data)

        status = "✅" if data.get("scrape_status") == "ok" else "⚠️"
        print(f"  {status} [{len(eng_rows)}/{len(ids_df)}] {r['book_title']}")

        new_since_checkpoint += 1
        if new_since_checkpoint >= CHECKPOINT_EVERY:
            save_checkpoint(eng_rows, ENG_OUT, "ENG", len(eng_rows))
            new_since_checkpoint = 0

        jitter_sleep()

    save_checkpoint(eng_rows, ENG_OUT, "ENG", len(eng_rows))
    print(f"[ENG] Phase 2 complete — {ENG_OUT}")

    # ── Summary ──
    eng_df = pd.read_csv(ENG_OUT)
    ok_count = (eng_df.get("scrape_status", pd.Series()) == "ok").sum()
    print(f"\n{'='*50}")
    print(f"DONE  |  IDs found: {len(ids_df)}  |  Engagement scraped: {ok_count}")
    print(f"{'='*50}")


if __name__ == "__main__":
    main()

Loaded 8396 books from pen_unique_books.csv
  ✅ [1/8396] Sloppy Firsts (Jessica Darling Series)
  ✅ [2/8396] Outlander (Outlander Series)
  ✅ [3/8396] Gender Queer: A Memoir
  ⚠️  Network error: HTTPSConnectionPool(host='www.goodreads.com', port=443): Read timed out. (read timeout=25)
  ⏳ backing off 12s (attempt 1/3)
  ❌ [4/8396] The Family Fletcher Takes Rock Island (Family Fletcher Series)
  ✅ [5/8396] Thirteen Reasons Why
  ✅ [6/8396] The Truth About Alice: A Novel
  ✅ [7/8396] Looking for Alaska
  ✅ [8/8396] Out of Darkness
  ✅ [9/8396] I Am Not Your Perfect Mexican Daughter
  ✅ [10/8396] Milk and Honey
  ✅ [11/8396] Life is Funny
  ✅ [12/8396] The Bluest Eye
  ✅ [13/8396] All Boys Aren't Blue
  ✅ [14/8396] The Absolutely True Diary of a Part-Time Indian
  ✅ [15/8396] Home at Last
  ✅ [16/8396] The Kite Runner
  ✅ [17/8396] Celebrate Your Body 2: The Ultimate Puberty Book for Preteen and Teen Girls
  ✅ [18/8396] Piecing Me Together
  ✅ [19/8396] Duels and Deception
  ✅ [20/8396] F